# Normalize biotic data by percent tree cover (SCI-TCC)

This script normalizes the raw mortality area biotic data by percent tree canopy cover. It uses the science-version of the NLCD's TCC product.

This script exports to Drive to avoid a 'Reprojection output too large' error. It does so by using shardSize, which appears to NOT work if exporting to Asset.

See StackExchange post submitted regarding the discrepancy here: https://gis.stackexchange.com/questions/501704/gee-inconsistent-shardsize-behavior-between-todrive-and-toasset

The data from 03_create_biotic_stack.R needs to be uploaded to GEE as an asset to run this script.

Note that data then needs to be re-uploaded as a GEE asset for use with script 05...

In [57]:
import ee

GEE_PROJECT = 'ee-tymc5571-multi-disturbance'
DRIVE_FOLDER = "GEE_Exports_one_offs"

ee.Initialize(project=GEE_PROJECT)


#########################
# DATA
#########################

ALL_BIOTIC_YEARS = list(range(1997, 2024))   # every band in the asset - for renaming only
RUN_YEARS = range(1997, 2024)                           # years to actually normalize; swap back to ALL_BIOTIC_YEARS for the full run

# rename on load so every band carries its year
biotic = (ee.Image("projects/ee-tymc5571-multi-disturbance/assets/biotic_gridded_1km_all_years_severity")
    .rename([f'biotic_{y}' for y in ALL_BIOTIC_YEARS])
)

# USFS TCC v2025-6, Science band (30 m, percent 0-100)
scitcc = (ee.ImageCollection("projects/gtac-data-publish/assets/TCC/Product_Version/2025-6")
    .filter(ee.Filter.eq('study_area', 'CONUS'))
    .select('Science_Percent_Tree_Canopy_Cover')
)

western_states = (ee.FeatureCollection("TIGER/2018/States")
    .filter(ee.Filter.inList('STUSPS', ['WA','OR','CA','ID','NV','MT','WY','UT','CO','AZ','NM']))
    .geometry()
)

# export straight onto the biotic grid so output pixels line up with the source
biotic_proj = biotic.select(0).projection()
BIOTIC_CRS = biotic_proj.crs().getInfo()
BIOTIC_TRANSFORM = biotic_proj.getInfo()['transform']
print("biotic grid:", BIOTIC_CRS, BIOTIC_TRANSFORM)

# Science TCC isn't post-processed, so non-tree areas carry a few percent of
# model noise instead of true zeros. Raise this (0.05-0.1) if sparse cells
# end up pinned at the 100 cap.
MIN_CANOPY_FRAC = 0.0
NODATA = -9999


#########################
# NORMALIZATION
#########################

def scitcc_frac(year: int) -> ee.Image:
    """Mean Science TCC canopy fraction (0..1) for one year. No reproject -
    reduceResolution aggregates to whatever grid the export asks for."""
    img = ee.Image(scitcc.filter(ee.Filter.calendarRange(year, year, 'year')).first())

    # masked 30 m pixels drop out of the mean rather than counting as zero
    return (img.divide(100) # get as a proportion
        .reduceResolution(reducer=ee.Reducer.mean(), maxPixels=4096)
    )


def biotic_normscitcc(year: int) -> ee.Image:
    """Biotic severity for `year` divided by the prior year's canopy fraction, capped at 100."""
    frac_prior = scitcc_frac(year - 1)

    # can't normalize where there was nothing (or too little) to normalize by
    frac_prior = frac_prior.updateMask(frac_prior.gt(MIN_CANOPY_FRAC))

    return (biotic.select(f'biotic_{year}')
        .toFloat()
        .divide(frac_prior) #Multiply forest fraction by 100 to transform to HA (100ha / 1km2) -- biotic mortility area is in HA; but then multiple proportion by 100 to get %, so evens out
        .min(100)
        .rename(f'biotic_normscitcc_{year}')
    )


# client-side loop keeps the band names exactly as written - no toBands() prefixes to strip
biotic_normscitcc_img = ee.Image.cat([biotic_normscitcc(y) for y in RUN_YEARS])

print("bands:", biotic.bandNames().getInfo())
print("bands:", biotic_normscitcc_img.bandNames().getInfo())




biotic grid: EPSG:5070 [1000, 0, -2375000, 0, -1000, 3189999]
bands: ['biotic_1997', 'biotic_1998', 'biotic_1999', 'biotic_2000', 'biotic_2001', 'biotic_2002', 'biotic_2003', 'biotic_2004', 'biotic_2005', 'biotic_2006', 'biotic_2007', 'biotic_2008', 'biotic_2009', 'biotic_2010', 'biotic_2011', 'biotic_2012', 'biotic_2013', 'biotic_2014', 'biotic_2015', 'biotic_2016', 'biotic_2017', 'biotic_2018', 'biotic_2019', 'biotic_2020', 'biotic_2021', 'biotic_2022', 'biotic_2023']
bands: ['biotic_normscitcc_1997', 'biotic_normscitcc_1998', 'biotic_normscitcc_1999', 'biotic_normscitcc_2000', 'biotic_normscitcc_2001', 'biotic_normscitcc_2002', 'biotic_normscitcc_2003', 'biotic_normscitcc_2004', 'biotic_normscitcc_2005', 'biotic_normscitcc_2006', 'biotic_normscitcc_2007', 'biotic_normscitcc_2008', 'biotic_normscitcc_2009', 'biotic_normscitcc_2010', 'biotic_normscitcc_2011', 'biotic_normscitcc_2012', 'biotic_normscitcc_2013', 'biotic_normscitcc_2014', 'biotic_normscitcc_2015', 'biotic_normscitcc_2016